In [ ]:
!pip install -q ezdxf

In [ ]:
import ezdxf

import json

from pathlib import Path

In [ ]:
# ======================================================
# Load State
# ======================================================


ROOT = Path.cwd().parent


STATE_FILE = (

ROOT /

"config" /

"project_state.json"

)



with open(

    STATE_FILE,

    encoding="utf-8"

) as f:


    PROJECT_STATE=json.load(f)



PROJECT_STATE

In [ ]:
# ======================================================
# Load Geometry
# ======================================================


with open(

    PROJECT_STATE["geometry_objects"],

    encoding="utf-8"

) as f:


    GEOMETRY_OBJECTS=json.load(f)



print(

"Geometry:",

len(GEOMETRY_OBJECTS)

)

In [ ]:
# ======================================================
# Load OCR Text
# ======================================================


with open(

    PROJECT_STATE["text_objects"],

    encoding="utf-8"

) as f:


    TEXT_OBJECTS=json.load(f)



print(

"Text:",

len(TEXT_OBJECTS)

)

In [ ]:
# ======================================================
# Create DXF Document
# ======================================================


doc = ezdxf.new(

    "R2018"

)


msp = doc.modelspace()



print(

"DXF Created"

)

In [ ]:
# ======================================================
# CAD Layers
# ======================================================


LAYERS={


"GEOMETRY":7,


"CIRCLE":3,


"TEXT":2,


"POLYLINE":4,


"ANNOTATION":1


}



for layer,color in LAYERS.items():


    doc.layers.add(

        layer,

        color=color

    )


print(
"Layers Ready"
)

In [ ]:
# ======================================================
# Add LINE
# ======================================================


for obj in GEOMETRY_OBJECTS:


    if obj["type"]=="LINE":


        msp.add_line(

            obj["start"],

            obj["end"],

            dxfattribs={

                "layer":
                "GEOMETRY"

            }

        )


print(
"Lines Added"
)

In [ ]:
# ======================================================
# Add Circle
# ======================================================


for obj in GEOMETRY_OBJECTS:


    if obj["type"]=="CIRCLE":


        msp.add_circle(

            obj["center"],

            obj["radius"],

            dxfattribs={

                "layer":
                "CIRCLE"

            }

        )


print(
"Circles Added"
)

In [ ]:
# ======================================================
# Add Polyline
# ======================================================


for obj in GEOMETRY_OBJECTS:


    if obj["type"]=="POLYLINE":


        msp.add_lwpolyline(

            obj["points"],

            dxfattribs={

                "layer":
                "POLYLINE"

            }

        )


print(
"Polyline Added"
)

In [ ]:
# ======================================================
# Add CAD TEXT
# ======================================================


for txt in TEXT_OBJECTS:


    text_entity=msp.add_text(

        txt["text"],

        dxfattribs={

            "height":

            max(

                txt["height"],

                5

            ),

            "layer":

            "TEXT"

        }

    )


    text_entity.dxf.insert=(

        txt["x"],

        txt["y"]

    )


print(

"Editable Text Added"

)

In [ ]:
# ======================================================
# Units
# ======================================================


doc.header["$INSUNITS"]=4


doc.header["$MEASUREMENT"]=1



print(
"MM Unit"
)

In [ ]:
# ======================================================
# Export DXF
# ======================================================


OUTPUT_DIR = ROOT / "output"


OUTPUT_DIR.mkdir(

    exist_ok=True

)



DXF_FILE = (

OUTPUT_DIR /

"AI_CAD_Final.dxf"

)



doc.saveas(

    DXF_FILE

)



print(

"Saved:",

DXF_FILE

)

In [ ]:
# ======================================================
# Verify
# ======================================================


try:

    test_doc=ezdxf.readfile(

        DXF_FILE

    )


    print(

    "DXF OK"

    )


except Exception as e:


    print(e)

In [ ]:
# ======================================================
# Update State
# ======================================================


PROJECT_STATE.update({

    "dxf_file":

    str(DXF_FILE)

})



with open(

STATE_FILE,

"w",

encoding="utf-8"

) as f:


    json.dump(

        PROJECT_STATE,

        f,

        indent=4

    )



print(
"Pipeline Complete"
)